# ============================================
### Title: Assignment 3.2
### Author: Doug Nolan
### Date: 22 June 2026
### Modified By: Doug Nolan
### Description: this program is a sentiment analysis and text preprocessor
# ============================================


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/doug/DSC 550/labeledTrainData.tsv", sep='\t')
#check to see data loaded as expected. 
print(df.head(10))

        id  sentiment                                             review
0   5814_8          1  With all this stuff going down at the moment w...
1   2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2   7759_3          0  The film starts with a manager (Nicholas Bell)...
3   3630_4          0  It must be assumed that those who praised this...
4   9495_8          1  Superbly trashy and wondrously unpretentious 8...
5   8196_8          1  I dont know why people think this is such a ba...
6   7166_2          0  This movie could have been very good, but come...
7  10633_1          0  I watched this video at a friend's house. I'm ...
8    319_1          0  A friend of mine bought this film for £1, and ...
9  8713_10          1  <br /><br />This movie is full of references. ...


In [3]:
print(df.shape)
# there are 25000 rows and 3 columns in this data


(25000, 3)


In [4]:
# find # of positive and negative reviews
# sentiment = 1 means positive , 0 means negative
sentiment = df['sentiment'].value_counts()
print(sentiment)

# this does the same as above - just less efficient - fun way to validate 
# count_pos = sentiment.get(1,0)
# count_neg = sentiment.get(0,0)

# print(count_pos)
# print(count_neg)

sentiment
1    12500
0    12500
Name: count, dtype: int64


# the ratings are split 50/50
# there are 12,500 negative and positve reviews

In [8]:
# use textblob to classify each movie as positive or negative
# assumption - positive scores = positive ... negative scores = negative
from textblob import TextBlob

def get_textblob_sentiment(text):
    # convert to str to handle blank or missing values
    analysis = TextBlob(str(text))

    if analysis.sentiment.polarity >= 0:
        return 1 #positive review
    else:
        return 0 # negative review

df['predicted_sentiment'] = df['review'].apply(get_textblob_sentiment)

print('Predicted Sentiment Counts:')
print(df['predicted_sentiment'].value_counts())

Predicted Sentiment Counts:
predicted_sentiment
1    19017
0     5983
Name: count, dtype: int64


In [9]:
# check accuracy of model - compare to original - 
# if better than 50%  - better than randomly guessing 

if 'sentiment' in df.columns:
    accuracy = (df['predicted_sentiment'] == df['sentiment']).mean()
    print(f'\nAccuracy compared to original data: {accuracy:.2%}')


Accuracy compared to original data: 68.52%


# Yes, the text blob is better than randomly guessing 50/50 on positive and negative reviews 


In [30]:
# rinse and repeat with another text sentiment analyzer 
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/doug/nltk_data...


True

In [35]:
sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    # vader outputs a dictionary of scores . compound is the overall score
    scores = sia.polarity_scores(str(text))
    return 1 if scores['compound'] >= 0 else 0 # 1 positve , 0 negative

df['vader_pred'] = df['review'].apply(get_vader_sentiment)

vader_accuracy = (df['vader_pred'] == df['sentiment']).mean()
print(f'VADER Model Accuracy: {vader_accuracy:.2f}')

if vader_accuracy >0.5:
    print('VADER model is better than randomly guessing')
else:
    print('VADER model is NOT better than randomly guessing')

VADER Model Accuracy: 0.69
VADER model is better than randomly guessing


In [36]:
# for fun, compare to text blob scores
vader_accuracy2 = (df['vader_pred'] == df['predicted_sentiment']).mean()
print(f'VADER Model Accuracy compared to TextBlob: {vader_accuracy:.2f}')

if vader_accuracy2 >0.5:
    print('VADER model is better than Text Blob')
else:
    print('VADER model is NOT better than Text Blob')

VADER Model Accuracy compared to TextBlob: 0.69
VADER model is better than Text Blob


# Part 2
## prepping text for a custom model

In [12]:
# import all the things
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


In [13]:
#only run once
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/doug/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
# conver to lower case
df['clean_review'] = df['review'].str.lower()

df[['review', 'clean_review']].head()

,review,clean_review
0,With all this stuff going down at the moment w...,with all this stuff going down at the moment w...
1,"\The Classic War of the Worlds\"" by Timothy Hi...","\the classic war of the worlds\"" by timothy hi..."
2,The film starts with a manager (Nicholas Bell)...,the film starts with a manager (nicholas bell)...
3,It must be assumed that those who praised this...,it must be assumed that those who praised this...
4,Superbly trashy and wondrously unpretentious 8...,superbly trashy and wondrously unpretentious 8...


In [15]:
# remove punctuation
def remove_special_characters(text):
    #remove html
    text = re.sub(r'<.*?>', '', text)

    # keep letters and spaces only
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text

df['clean_review'] = df['clean_review'].apply(remove_special_characters)

df['clean_review'].head()

0    with all this stuff going down at the moment w...
1    the classic war of the worlds by timothy hines...
2    the film starts with a manager nicholas bell g...
3    it must be assumed that those who praised this...
4    superbly trashy and wondrously unpretentious s...
Name: clean_review, dtype: str

In [20]:
# remove stop words
# already downloaded the stop words ^^
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return ' '.join(filtered_words)

df['clean_review'] = df['clean_review'].apply(remove_stopwords)

df['clean_review'].head()

0    stuff going moment mj ive started listening mu...
1    classic war worlds timothy hines entertaining ...
2    film starts manager nicholas bell giving welco...
3    must assumed praised film greatest filmed oper...
4    superbly trashy wondrously unpretentious explo...
Name: clean_review, dtype: str

In [22]:
# apply NLTK PorterStemmer
stemmer = PorterStemmer()

def stem_text(text):
    words = text.split()
    stemmed_words = [
        stemmer.stem(word)
        for word in words]
    return ' '.join(stemmed_words)

df['stemmed_review'] = df['clean_review'].apply(stem_text)

df[['clean_review', 'stemmed_review']].head() 

,clean_review,stemmed_review
0,stuff going moment mj ive started listening mu...,stuff go moment mj ive start listen music watc...
1,classic war worlds timothy hines entertaining ...,classic war world timothi hine entertain film ...
2,film starts manager nicholas bell giving welco...,film start manag nichola bell give welcom inve...
3,must assumed praised film greatest filmed oper...,must assum prais film greatest film opera ever...
4,superbly trashy wondrously unpretentious explo...,superbl trashi wondrous unpretenti exploit hoo...


In [23]:
# create bag-of-words matrix
# row = movie review 
# column = vocab word

bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(df['stemmed_review'])

print('Bag of Words Matrix Shape:')
print(bow_matrix.shape)

Bag of Words Matrix Shape:
(25000, 108798)


In [24]:
# check out words from the matrix
vocab = bow_vectorizer.get_feature_names_out()

print('First 20 vocab words:')
print(vocab[:20])

First 20 vocab words:
['aa' 'aaa' 'aaaaaaah' 'aaaaah' 'aaaaatchkah' 'aaaahhhhhhh' 'aaaand'
 'aaaarrgh' 'aaah' 'aaand' 'aaargh' 'aaarghhow' 'aaaugh' 'aachen' 'aada'
 'aadha' 'aadmittedli' 'aag' 'aaghh' 'aah']


In [26]:
# create tf-idf matrix
# purposely mis spelling this - QoL to type shorter / easier to spell things
tf_vector = TfidfVectorizer()

tf_matrix = tf_vector.fit_transform(df['stemmed_review'])

print('TF-IDF Matrix Shape:')
print(tf_matrix.shape)

TF-IDF Matrix Shape:
(25000, 108798)


In [28]:
#compare both matrices to see if the word count is the same

print('Bag of Words Shape :', bow_matrix.shape)
print('TF-IDF Shape       :', tf_matrix.shape)

if bow_matrix.shape == tf_matrix.shape:
    print('Both matrices have matching dimensions.')
else:
    print('Dimensions do not match.')

Bag of Words Shape : (25000, 108798)
TF-IDF Shape       : (25000, 108798)
Both matrices have matching dimensions.
